In this notebook I checked for orientation for all data and their segments

In [1]:
import os
import nibabel as nib
import glob


In [2]:

BASE_DIR = os.path.dirname(os.path.abspath(os.getcwd()))

RAW_DATA_DIR = os.path.join(BASE_DIR, '/workspace/data')
# RAW_DATA_DIR = os.path.join("/workspace/",RAW_DATA_DIR)
print(RAW_DATA_DIR)
IMAGES_SOURCE = os.path.join(RAW_DATA_DIR, 'ao_imgs')
LABELS_SOURCE = os.path.join(RAW_DATA_DIR, 'ao_segs')

print(f"Source images: {IMAGES_SOURCE}")
print(f"Source labels: {LABELS_SOURCE}")


/workspace/data
Source images: /workspace/data/ao_imgs
Source labels: /workspace/data/ao_segs


# check orientations

In [3]:

from collections import Counter


img_orient_counter = Counter()
lbl_orient_counter = Counter()

for lbl_name in os.listdir(LABELS_SOURCE):
    if not lbl_name.endswith(".nii.gz"):
        continue

    img_name = lbl_name.replace(".nii.gz", "_0000.nii.gz")
    img_path = os.path.join(IMAGES_SOURCE, img_name)
    lbl_path = os.path.join(LABELS_SOURCE, lbl_name)

    if not os.path.exists(img_path):
        continue

    img = nib.load(img_path)
    lbl = nib.load(lbl_path)

    img_orient = nib.aff2axcodes(img.affine)
    lbl_orient = nib.aff2axcodes(lbl.affine)

    img_orient_counter[img_orient] += 1
    lbl_orient_counter[lbl_orient] += 1


print("=== IMAGE ORIENTATIONS ===")
print(f"RAS count     : {img_orient_counter[('R','A','S')]}")
print(f"Non‑RAS count : {sum(img_orient_counter.values()) - img_orient_counter[('R','A','S')]}")
print("Breakdown:", dict(img_orient_counter))

print("\n=== LABEL ORIENTATIONS ===")
print(f"RAS count     : {lbl_orient_counter[('R','A','S')]}")
print(f"Non‑RAS count : {sum(lbl_orient_counter.values()) - lbl_orient_counter[('R','A','S')]}")
print("Breakdown:", dict(lbl_orient_counter))


=== IMAGE ORIENTATIONS ===
RAS count     : 744
Non‑RAS count : 0
Breakdown: {('R', 'A', 'S'): 744}

=== LABEL ORIENTATIONS ===
RAS count     : 744
Non‑RAS count : 0
Breakdown: {('R', 'A', 'S'): 744}


# unify orientations to RAS for all images ans their labels

In [4]:



imgs_dir = r"D:\projects\4D-flow-MRI-aortic-segmentation-\data\ao_imgs"
segs_dir = r"D:\projects\4D-flow-MRI-aortic-segmentation-\data\ao_segs"


def orient_to_ras_inplace(nii_path):
    img = nib.load(nii_path)
    axcodes = nib.aff2axcodes(img.affine)

    if axcodes == ('R', 'A', 'S'):
        print(f"[OK ] Already RAS: {os.path.basename(nii_path)}")
        return

    print(f"[FIX] Reorienting {os.path.basename(nii_path)} from {axcodes} → RAS")
    img_ras = nib.as_closest_canonical(img)
    nib.save(img_ras, nii_path)


# --- Process images ---
for img_path in glob.glob(os.path.join(imgs_dir, "*.nii*")):
    orient_to_ras_inplace(img_path)

# --- Process labels ---
for seg_path in glob.glob(os.path.join(segs_dir, "*.nii*")):
    orient_to_ras_inplace(seg_path)

print("✅ Orientation harmonization complete.")


✅ Orientation harmonization complete.


# test

In [10]:

from collections import Counter


img_orient_counter = Counter()
lbl_orient_counter = Counter()

for lbl_name in os.listdir(segs_dir):
    if not lbl_name.endswith(".nii.gz"):
        continue

    img_name = lbl_name.replace(".nii.gz", "_0000.nii.gz")
    img_path = os.path.join(imgs_dir, img_name)
    lbl_path = os.path.join(segs_dir, lbl_name)

    if not os.path.exists(img_path):
        continue

    img = nib.load(img_path)
    lbl = nib.load(lbl_path)

    img_orient = nib.aff2axcodes(img.affine)
    lbl_orient = nib.aff2axcodes(lbl.affine)

    img_orient_counter[img_orient] += 1
    lbl_orient_counter[lbl_orient] += 1


print("=== IMAGE ORIENTATIONS ===")
print(f"RAS count     : {img_orient_counter[('R','A','S')]}")
print(f"Non‑RAS count : {sum(img_orient_counter.values()) - img_orient_counter[('R','A','S')]}")
print("Breakdown:", dict(img_orient_counter))

print("\n=== LABEL ORIENTATIONS ===")
print(f"RAS count     : {lbl_orient_counter[('R','A','S')]}")
print(f"Non‑RAS count : {sum(lbl_orient_counter.values()) - lbl_orient_counter[('R','A','S')]}")
print("Breakdown:", dict(lbl_orient_counter))


=== IMAGE ORIENTATIONS ===
RAS count     : 744
Non‑RAS count : 0
Breakdown: {('R', 'A', 'S'): 744}

=== LABEL ORIENTATIONS ===
RAS count     : 744
Non‑RAS count : 0
Breakdown: {('R', 'A', 'S'): 744}


# Resampling

In [5]:
import SimpleITK as sitk

IMAGES_TR = os.path.join(RAW_DATA_DIR, 'ao_imgs')
LABELS_TR = os.path.join(RAW_DATA_DIR, 'ao_segs')

fixed = 0

for lbl in os.listdir(LABELS_TR):
    if not lbl.endswith(".nii.gz"):
        continue

    img_name = lbl.replace(".nii.gz", "_0000.nii.gz")
    img_path = os.path.join(IMAGES_TR, img_name)
    lbl_path = os.path.join(LABELS_TR, lbl)

    if not os.path.exists(img_path):
        continue

    img = sitk.ReadImage(img_path)
    seg = sitk.ReadImage(lbl_path)

    if img.GetSpacing() != seg.GetSpacing() \
       or img.GetSize() != seg.GetSize() \
       or img.GetDirection() != seg.GetDirection():

        resampler = sitk.ResampleImageFilter()
        resampler.SetReferenceImage(img)
        resampler.SetInterpolator(sitk.sitkNearestNeighbor)
        resampler.SetTransform(sitk.Transform())
        resampler.SetDefaultPixelValue(0)

        seg_fixed = resampler.Execute(seg)
        sitk.WriteImage(seg_fixed, lbl_path)

        fixed += 1
        # print(f"Fixed geometry for {lbl}")

print(f"\nTotal fixed segmentations: {fixed}")


Total fixed segmentations: 124


# keep the 599 images and labels for the verified pts

In [10]:
BASE_DIR = os.path.dirname(os.path.abspath(os.getcwd()))

SHEET_DIR = os.path.join(BASE_DIR, '/workspace/preprocessing/HVOLS_599.csv')
SHEET_DIR

'/workspace/preprocessing/HVOLS_599.csv'

In [14]:
import pandas as pd

df = pd.read_csv(SHEET_DIR)
df = df["id"]


In [15]:
import os
import re

def filter_files_by_ids(folder_path, valid_ids):
    """
    Delete files whose patient ID is NOT in valid_ids.

    Args:
        folder_path (str): directory containing .nii.gz files
        valid_ids (set or list): allowed patient IDs (e.g., [12, 274, ...])
    """

    # Convert to set for fast lookup
    valid_ids = set(map(int, valid_ids))

    # Regex to extract ID from filenames
    pattern = re.compile(r"AORTA_(\d+)")

    deleted = 0
    kept = 0

    for fname in os.listdir(folder_path):
        if not fname.endswith(".nii.gz"):
            continue

        match = pattern.search(fname)
        if not match:
            print(f"[WARNING] Skipping unrecognized file: {fname}")
            continue

        patient_id = int(match.group(1))

        file_path = os.path.join(folder_path, fname)

        if patient_id not in valid_ids:
            os.remove(file_path)
            deleted += 1
        else:
            kept += 1

    print(f"✅ Done. Kept: {kept}, Deleted: {deleted}")



IMAGES_TR = os.path.join(RAW_DATA_DIR, 'ao_imgs')
LABELS_TR = os.path.join(RAW_DATA_DIR, 'ao_segs')
filter_files_by_ids(IMAGES_TR, list(df))
filter_files_by_ids(LABELS_TR, list(df))

✅ Done. Kept: 599, Deleted: 145
✅ Done. Kept: 599, Deleted: 145


In [16]:

num_files = len([
    f for f in os.listdir(IMAGES_TR)
    if f.endswith(".nii.gz")
])

print(f"Number of NIfTI files: {num_files}")


Number of NIfTI files: 599


In [17]:
num_files = len([
    f for f in os.listdir(LABELS_TR)
    if f.endswith(".nii.gz")
])

print(f"Number of NIfTI files: {num_files}")


Number of NIfTI files: 599
